<a href="https://colab.research.google.com/github/phuc11731510/chess_variant_engine/blob/mcts-capacity-256/FairyZero_vanhanh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FairyZero — sổ tay vận hành

Sinh dữ liệu · huấn luyện · arena · tải về máy. **Không biên dịch lại mỗi phiên.**

| Mục | Việc | Thời gian |
|---|---|---|
| 1 | Khởi động (tải binary dựng sẵn + ONNX Runtime) | ~1 phút |
| 1b | *(chỉ khi cần)* biên dịch lại từ đầu | 8-12 phút |
| 2 | Tạo mạng đời 0 | ~30 giây |
| 3 | Sinh dữ liệu huấn luyện | theo `--max-seconds` |
| 4 | Gom + tải dữ liệu về máy | vài phút |
| 5 | Huấn luyện đời sau | 10-40 phút |
| 6 | Arena so hai đời | ~30 phút |

**Runtime → Change runtime type → T4 GPU** trước khi chạy.


## 0. Kiểm tra GPU


In [1]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv


/bin/bash: line 1: nvidia-smi: command not found


## 1. Khởi động — KHÔNG biên dịch

Tải mã nguồn (nhỏ), lấy **binary dựng sẵn** từ GitHub Release, tải ONNX Runtime,
rồi sinh `run.sh`. Phải thấy `[quick] OK`.

Sinh `run.sh` là bắt buộc: mỗi cell Colab là một shell mới nên `export` không
sống sót; engine cần `LD_LIBRARY_PATH` trỏ tới ONNX Runtime và các thư viện CUDA.


In [2]:
%cd /content
!rm -rf chess_variant_engine
!git clone -q --depth 1 -b mcts-capacity-256 https://github.com/phuc11731510/chess_variant_engine.git
E = "/content/chess_variant_engine/custom_engine"

# Khoi dong KHONG bien dich: tai binary dung san tu Release + ONNX Runtime.
# Neu Release chua co binary (lan dau), cell nay se bao loi -- chay muc 1b.
!BIN_URL=https://github.com/phuc11731510/chess_variant_engine/releases/download/v1.0.0/custom_engine bash {E}/scripts/colab_quickstart.sh


/content
[quick] tai binary: https://github.com/phuc11731510/chess_variant_engine/releases/download/v1.0.0/custom_engine
[quick] FATAL: tai binary that bai


## 1b. Biên dịch lại — chỉ khi cần

Chạy mục này khi: Release **chưa có** binary, Colab **đổi base image** (mục 1 báo lỗi),
hoặc bạn **vừa sửa mã nguồn**.

Cuối cell nó tải binary về máy bạn — đưa lên GitHub Release với tên đúng
`custom_engine` thì các phiên sau chỉ cần mục 1.


In [3]:
# CHI CHAY KHI CAN: bien dich tu dau (~8-12 phut).
# Can khi: Release chua co binary, hoac Colab doi base image, hoac ban vua sua ma nguon.
E = "/content/chess_variant_engine/custom_engine"
!bash {E}/scripts/colab_setup.sh 2>&1 | tail -4
!bash {E}/scripts/colab_prebuilt.sh wrap
print()
print("Binary moi o:", E + "/build-linux/custom_engine")
print("Tai ve roi dua len GitHub Release (ten dung: custom_engine) de lan sau khoi bien dich.")
from google.colab import files
files.download(E + "/build-linux/custom_engine")


      --provider cuda --fixed-batch 32 --weights /content/chess_variant_engine/custom_engine/python/seed.onnx --out /content/chess_variant_engine/custom_engine/python/games_gen0
[colab] 2) Train next generation (warm-start from the gen-0 .pt):
  python /content/chess_variant_engine/custom_engine/python/train.py --data /content/chess_variant_engine/custom_engine/python/games_gen0 --epochs 20 --batch 1024 \
      --amp --init-from /content/chess_variant_engine/custom_engine/python/seed.pt --out /content/chess_variant_engine/custom_engine/python/model_gen1.onnx
[prebuilt] engine dir: /content/chess_variant_engine/custom_engine
[prebuilt] cache dir:  /content/drive/MyDrive/FairyZero_prebuilt
[prebuilt] ORT pkg:    onnxruntime-linux-x64-gpu-1.20.1
[prebuilt] wrote wrapper: /content/chess_variant_engine/custom_engine/run.sh
[prebuilt] health check: --test-uci ...
[prebuilt] OK — engine runs on this Colab image. SKIP colab_setup.sh.
[prebuilt] Run the engine via:  bash /content/chess_variant_

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 2. Mạng đời 0

Trọng số ngẫu nhiên theo **seed cố định `5523225413`** → tái lập được y hệt.

Kiến trúc `144 × 12` SE-8 **phải giữ nguyên suốt chuỗi warm-start** — đổi giữa chừng
thì không nạp được trọng số đời trước.


In [4]:
# Mang doi 0: trong so ngau nhien theo seed co dinh -> tai lap duoc.
E = "/content/chess_variant_engine/custom_engine"
SEED = 5523225413
!cd {E}/python && python make_seed.py --seed {SEED} --channels 144 --blocks 12 --se-ratio 8 \
    --out /content/gen0.onnx
!ls -la /content/gen0.*

[seed] 5.49M params -> saved /content/gen0.pt
[export] wrote /content/gen0.onnx
[verify] inputs=[('input', ['batch', 226, 10, 10])]
[verify] outputs=[('policy', ['batch', 10600]), ('value', ['batch', 3])]
[verify] runtime OK: policy(2, 10600) value(2, 3) value_sum=[1.        0.9999999]
[verify] PASS: ONNX I/O contract matches engine (policy=logits, value=softmax WDL).
[seed] gen-0 bootstrap ready: /content/gen0.onnx
-rw-r--r-- 1 root root 22033052 Sep 22 07:11 /content/gen0.onnx
-rw-r--r-- 1 root root 22058485 Sep 22 07:11 /content/gen0.pt


In [5]:
# Tải mạng đời 0 về máy
from google.colab import files
import time
for f in ["/content/gen0.onnx", "/content/gen0.pt"]:
    print("tai:", f)
    files.download(f)
    time.sleep(1)

tai: /content/gen0.onnx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

tai: /content/gen0.pt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 3. Sinh dữ liệu huấn luyện

Cấu hình dưới đây **đã được đo đạc xác nhận** là tốt nhất trên T4, không phải phỏng đoán:

- `--fixed-batch 16` — MCTS gom trung bình ~15 lá/lượt nên batch 16 lấp đầy **94%**,
  chỉ 5,7% phí padding. Đặt 64 chỉ lấp 62% → 38% phí, **chậm hơn**.
- `--parallel 4` — quét 1→32 cho throughput gần như phẳng; 4 cho `eval/giây` cao nhất.
- **Không** `--batch-aggregate` — đo được là ngang hoặc tệ hơn.

⚠ `--max-seconds` dừng **mềm**: ngừng nhận ván mới nhưng ván đang chạy vẫn hoàn tất.
Với `--parallel 4` sẽ vượt giờ khoảng 2-3 phút — đừng ngắt sớm, nếu không mất dữ liệu.

Đọc `--- Throughput ---` ở cuối để theo dõi tốc độ. **So cấu hình bằng `NN eval/giay`,
đừng bằng `Van/gio`** (với mẫu nhỏ, độ dài ván dao động áp đảo).


In [ ]:
E = "/content/chess_variant_engine/custom_engine"
WEIGHTS = "/content/gen0.onnx"      # doi thanh doi moi nhat cho cac gen sau
OUT     = "/content/games_gen0"
SECS    = 19080                          # ~5.3 gio; tru vai phut cho quota Colab

# Cau hinh nay da duoc DO DAC xac nhan la tot nhat tren T4 (xem HUONG_DAN.md B.2):
#   --parallel 4 --fixed-batch 16   ->  lap day 94% moi batch, chi 5.7% phi padding
#   KHONG dung --batch-aggregate    ->  do duoc la ngang hoac te hon
!bash {E}/run.sh --selfplay \
    --games 100000 --max-seconds {SECS} \
    --visits 800 --max-moves 400 --temp-cutoff 32 \
    --parallel 4 --provider cuda --fixed-batch 16 \
    --noise-alpha 0.15 --show-nps \
    --weights {WEIGHTS} --out {OUT}


## 4. Gom và tải dữ liệu huấn luyện về máy

Hàng nghìn tệp `.gz` nhỏ → gom thành **một** `.zip` rồi tải thẳng về máy.
`train.py` đọc được `.zip` trực tiếp, không cần giải nén.

Không đi qua Drive: Drive FUSE rất chậm với nhiều tệp nhỏ, và bạn muốn giữ bản gốc ở máy.


In [ ]:
# Gom .gz thanh MOT file zip roi tai THANG ve may (khong qua Drive).
E   = "/content/chess_variant_engine/custom_engine"
OUT = "/content/games_gen0"
ZIP = "/content/games_gen0.zip"

!python {E}/python/archive.py pack {OUT} --out {ZIP}
!ls -la {ZIP}

from google.colab import files
files.download(ZIP)     # trinh duyet se hoi noi luu


## 5. Huấn luyện đời sau

Warm-start từ `.pt` của đời trước. Muốn dùng **cửa sổ trượt nhiều đời** thì truyền
nhiều zip ngăn bằng dấu phẩy: `--data "gen0.zip, gen1.zip, gen2.zip"`.

Nếu dữ liệu đang ở máy bạn, bỏ dấu `#` ở dòng `files.upload()` để tải lên trước.


In [ ]:
E = "/content/chess_variant_engine/custom_engine"

# Neu du lieu dang o may ban thi tai len truoc (chay cell nay roi chon file .zip):
# from google.colab import files; files.upload()

DATA = "/content/games_gen0.zip"          # nhieu doi: "a.zip, b.zip, c.zip"
INIT = "/content/seed_gen0.pt"            # warm-start tu doi truoc
OUT  = "/content/12bx144fx8s_gen1.onnx"
OUT_PT = OUT.replace(".onnx", ".pt")

# --channels/--blocks PHAI giu nguyen suot chuoi warm-start.
!python {E}/python/train.py \
    --data "{DATA}" --init-from {INIT} \
    --epochs 2 --batch 1024 --lr 1e-3 --amp \
    --q-ratio 0.2 --weight-decay 1e-4 --report-every 20 \
    --channels 144 --blocks 12 \
    --out {OUT} --seed 48
!ls -la {OUT} {OUT_PT}


### Tải mạng vừa huấn luyện về máy


In [ ]:
from google.colab import files
import time
for f in ["/content/gen0.onnx", "/content/gen0.pt"]:
    print("tai:", f)
    files.download(f)
    time.sleep(1)


## 6. Arena — đời mới có thật sự mạnh hơn không

⚠ **20 ván là quá ít**: sai số khoảng **±150 Elo**, tức đời mới có thể mạnh hơn 200 Elo
hoặc yếu hơn 100 Elo mà bạn vẫn thấy cùng một kết quả.

Để phát hiện chênh lệch ~50 Elo cần **400-1000 ván**. Cùng ngân sách GPU thì
**400 ván × 200 visits** cho nhiều thông tin hơn hẳn 20 ván × 800 visits.


In [ ]:
E = "/content/chess_variant_engine/custom_engine"

# Doi moi vs doi cu. LUU Y ve thong ke: 20 van cho sai so ~±150 Elo -- qua lon de
# ket luan. De phat hien chenh ~50 Elo can 400-1000 van, nen HA visits xuong va
# TANG so van: cung ngan sach GPU, 400 van x 200 visits cho nhieu thong tin hon
# han 20 van x 800 visits.
!bash {E}/run.sh --arena \
    --model-a /content/12bx144fx8s_gen1.onnx \
    --model-b /content/seed_gen0.onnx \
    --games 400 --visits 200 --temp-cutoff 32 \
    --provider cuda --fixed-batch 16 --max-moves 400 --show-nps


---
## Vòng lặp một đời

```
muc 1  ->  muc 3 (sinh)  ->  muc 4 (tai ve)  ->  muc 5 (train)  ->  muc 6 (arena)
                ^                                        |
                +---- mang moi lam --weights -----------+
```

Mỗi đời: đổi `WEIGHTS` ở mục 3 sang mạng mới nhất, đổi `OUT` sang `games_genN`,
và ở mục 5 đổi `INIT` sang `.pt` của đời trước.

Tham khảo đầy đủ mọi cờ: `custom_engine/HUONG_DAN.md` mục D.
